# UKRI FoR Prediction Analysis

This notebook analyses the output of the FoR classification pipeline **without rerunning either model**.

It combines:

- the original historical input Parquet files from `prod_data_temp/`
- the consolidated FoR prediction Parquet output from `data/output/`

into a single analysis dataset.

## What this analysis is intended to answer

The main questions are:

1. **Coverage:** What proportion of applications were classified by the primary Group model, the fallback Division model, or neither?
2. **Multi-label behaviour:** How many categories are assigned to each application?
3. **Score behaviour:** Are selected model scores generally strong or weak, and which categories have weaker score distributions?
4. **Text quality:** Are fallback/unresolved applications associated with shorter or missing application text?
5. **Source effects:** Does `ApplicationOriginSource` have different classification/fallback/null rates?
6. **Category concentration:** Which FoR Groups and Divisions dominate the predictions, and how much of the taxonomy is being used?
7. **Rare categories:** Which categories are predicted very infrequently and may need review?
8. **Common multi-label combinations:** Which Groups tend to be predicted together?
9. **Potential anomalies:** Are there applications with unusually many predictions, inconsistent null fields, invalid category formats, or duplicate outputs?
10. **Completeness:** Does every unique input application appear in the final prediction output?

The notebook intentionally does **not** compare the ten historical files against each other because they are simply chunks of the same backfill population.

## 1. Imports and project paths

The notebook assumes the same project structure used for the deployment work:

```text
deployment_FoR_Classifier/
├── prod_data_temp/
├── data/
│   └── output/
├── notebooks/
└── ...
```

If the prediction file is not the latest `*_FoR.parquet` file under `data/output/`, set `PREDICTION_FILE` explicitly in the next section.

In [ ]:
from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "prod_data_temp").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "prod_data_temp").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR.parent

INPUT_DIR = PROJECT_ROOT / "prod_data_temp"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
ANALYSIS_OUTPUT_DIR = PROJECT_ROOT / "data" / "analysis"

ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 2. Configuration

For the one-off historical backfill:

- the input population is all local Parquet files under `prod_data_temp/`
- the prediction output is the consolidated FoR Parquet file produced by the backfill notebook

By default, this notebook selects the most recently modified file ending in `_FoR.parquet`.

If that is not the correct file, set `PREDICTION_FILE` manually.

In [ ]:
INPUT_PATTERN = "*.parquet"

# Leave as None to automatically select the newest *_FoR.parquet file.
PREDICTION_FILE = None

ID_FIELDS = [
    "ApplicationID",
    "ApplicationOriginSource",
]

TEXT_FIELDS = [
    "ApplicationTitle",
    "ApplicationSummary",
]

EXPECTED_OUTPUT_COLUMNS = [
    "ApplicationID",
    "ApplicationOriginSource",
    "model_run_date",
    "category_id",
    "score_type",
    "score",
    "Taxonomy",
    "model_name",
    "model_version",
]

# Useful thresholds for diagnostics only.
# These are NOT model decision thresholds.
LOW_SCORE_REVIEW_CUTOFF = 0.50
HIGH_PREDICTION_COUNT = 6
RARE_CATEGORY_MAX_APPLICATIONS = 5

## 3. Load and combine the original input files

### Why this matters

The prediction output contains one row per predicted category, so it cannot by itself tell us:

- whether an application had a title or summary
- how much text the model received
- which source system the application came from
- whether every original application survived into the output

We therefore reconstruct the unique input population first.

### Deduplication

The agreed application identity is:

`ApplicationID + ApplicationOriginSource`

If the same application appears more than once across the historical files, the first occurrence is retained for analysis so that the analysis population matches the deployment deduplication rule.

In [ ]:
input_files = sorted(INPUT_DIR.glob(INPUT_PATTERN))

if not input_files:
    raise FileNotFoundError(
        f"No Parquet files found under {INPUT_DIR}"
    )

print("Input files found:", len(input_files))

input_frames = []

for file_path in input_files:
    df_part = pd.read_parquet(file_path, engine="pyarrow")
    df_part["_source_file"] = file_path.name
    input_frames.append(df_part)

df_input_raw = pd.concat(
    input_frames,
    ignore_index=True,
)

required_input_columns = ID_FIELDS + TEXT_FIELDS
missing_input_columns = [
    c for c in required_input_columns
    if c not in df_input_raw.columns
]

if missing_input_columns:
    raise ValueError(
        f"Input data missing required columns: {missing_input_columns}"
    )

raw_input_rows = len(df_input_raw)

df_input = (
    df_input_raw
    .drop_duplicates(
        subset=ID_FIELDS,
        keep="first",
    )
    .reset_index(drop=True)
)

print("Raw input rows:", raw_input_rows)
print("Unique applications:", len(df_input))
print("Duplicate rows removed:", raw_input_rows - len(df_input))

## 4. Load the consolidated prediction output

### Why this matters

This is the actual production-shaped model result that stakeholders will consume.

The output can contain:

- multiple rows for one application when the primary model assigns multiple Groups
- a 2-digit Division prediction where the fallback model was required
- one row with native null `category_id`, `score_type`, and `score` where the application could not be classified

We validate the expected schema before beginning analysis.

In [ ]:
if PREDICTION_FILE is None:
    prediction_candidates = sorted(
        OUTPUT_DIR.glob("*_FoR.parquet"),
        key=lambda p: p.stat().st_mtime,
    )

    if not prediction_candidates:
        raise FileNotFoundError(
            f"No *_FoR.parquet prediction file found under {OUTPUT_DIR}"
        )

    prediction_path = prediction_candidates[-1]
else:
    prediction_path = Path(PREDICTION_FILE)

    if not prediction_path.is_absolute():
        prediction_path = PROJECT_ROOT / prediction_path

if not prediction_path.exists():
    raise FileNotFoundError(prediction_path)

df_predictions = pd.read_parquet(
    prediction_path,
    engine="pyarrow",
)

missing_prediction_columns = [
    c for c in EXPECTED_OUTPUT_COLUMNS
    if c not in df_predictions.columns
]

if missing_prediction_columns:
    raise ValueError(
        "Prediction file missing expected columns: "
        f"{missing_prediction_columns}"
    )

print("Prediction file:", prediction_path)
print("Prediction rows:", len(df_predictions))
print(
    "Unique applications in predictions:",
    df_predictions[ID_FIELDS].drop_duplicates().shape[0],
)
display(df_predictions.head())

## 5. Build a single prediction-level analysis dataframe

### What we are doing

We left-join the prediction output back to the original input population.

The resulting `df_analysis` remains **prediction-level**:

- one application with three Group predictions appears on three rows
- one application with one fallback Division appears on one row
- an unclassified application appears once with null prediction fields

### What this tells us

This dataframe allows us to analyse model behaviour alongside the exact text fields and source metadata that were present at inference time.

In [ ]:
analysis_input_columns = list(dict.fromkeys(
    ID_FIELDS
    + TEXT_FIELDS
    + [
        c for c in df_input.columns
        if c not in ID_FIELDS + TEXT_FIELDS
        and c != "_source_file"
    ]
))

df_analysis = df_input[
    [c for c in analysis_input_columns if c in df_input.columns]
].merge(
    df_predictions,
    on=ID_FIELDS,
    how="left",
    validate="one_to_many",
)

print("Prediction-level analysis rows:", len(df_analysis))
print(
    "Unique analysis applications:",
    df_analysis[ID_FIELDS].drop_duplicates().shape[0],
)

## 6. Add derived analytical fields

### Why these fields are useful

The raw model output tells us *what* was predicted, but not how difficult an application may have been to classify.

We derive:

- whether title/summary were available
- title and summary word counts
- total text length
- whether the prediction came from the Group model, fallback Division model, or remained unresolved
- number of categories assigned to each application

### Prediction level inference

Because the final stakeholder output does not include an internal `prediction_source` field:

- a **4-digit** category is interpreted as a primary Group prediction
- a **2-digit** category is interpreted as a fallback Division prediction
- null `category_id` is interpreted as unresolved

In [ ]:
def clean_nullable_text(series):
    return (
        series
        .astype("string")
        .fillna("")
        .str.strip()
    )

title_text = clean_nullable_text(
    df_analysis["ApplicationTitle"]
)
summary_text = clean_nullable_text(
    df_analysis["ApplicationSummary"]
)

df_analysis["has_title"] = title_text.ne("")
df_analysis["has_summary"] = summary_text.ne("")
df_analysis["has_any_text"] = (
    df_analysis["has_title"]
    | df_analysis["has_summary"]
)

df_analysis["title_word_count"] = (
    title_text
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

df_analysis["summary_word_count"] = (
    summary_text
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

df_analysis["total_text_word_count"] = (
    df_analysis["title_word_count"]
    + df_analysis["summary_word_count"]
)

category_string = (
    df_analysis["category_id"]
    .astype("string")
)

df_analysis["prediction_level"] = np.select(
    [
        category_string.str.fullmatch(r"\d{4}", na=False),
        category_string.str.fullmatch(r"\d{2}", na=False),
        category_string.isna(),
    ],
    [
        "PRIMARY_GROUP",
        "FALLBACK_DIVISION",
        "UNRESOLVED",
    ],
    default="OTHER",
)

prediction_count = (
    df_analysis
    .loc[df_analysis["category_id"].notna()]
    .groupby(ID_FIELDS)
    .size()
    .rename("prediction_count")
)

df_analysis = df_analysis.merge(
    prediction_count,
    on=ID_FIELDS,
    how="left",
)

df_analysis["prediction_count"] = (
    df_analysis["prediction_count"]
    .fillna(0)
    .astype(int)
)

display(
    df_analysis[
        ID_FIELDS
        + [
            "prediction_level",
            "prediction_count",
            "total_text_word_count",
            "category_id",
            "score",
        ]
    ].head(10)
)

## 7. Create one-row-per-application summary dataframe

### Why this is important

Many analyses should be performed at **application level**, not prediction-row level.

For example, if one application has four Group predictions, it should still count as **one classified application** when measuring classification coverage.

`df_application_summary` therefore contains exactly one row per:

`ApplicationID + ApplicationOriginSource`

and includes:

- text availability
- text length
- prediction count
- classification route
- maximum and mean selected score

In [ ]:
application_base = (
    df_analysis[
        ID_FIELDS
        + [
            "ApplicationTitle",
            "ApplicationSummary",
            "has_title",
            "has_summary",
            "has_any_text",
            "title_word_count",
            "summary_word_count",
            "total_text_word_count",
        ]
    ]
    .drop_duplicates(subset=ID_FIELDS)
)

prediction_agg = (
    df_analysis
    .groupby(ID_FIELDS, dropna=False)
    .agg(
        prediction_count=("prediction_count", "max"),
        max_score=("score", "max"),
        mean_score=("score", "mean"),
        has_primary_group=(
            "prediction_level",
            lambda s: (s == "PRIMARY_GROUP").any(),
        ),
        has_fallback_division=(
            "prediction_level",
            lambda s: (s == "FALLBACK_DIVISION").any(),
        ),
        is_unresolved=(
            "category_id",
            lambda s: s.isna().all(),
        ),
    )
    .reset_index()
)

df_application_summary = application_base.merge(
    prediction_agg,
    on=ID_FIELDS,
    how="left",
)

df_application_summary["classification_status"] = np.select(
    [
        df_application_summary["has_primary_group"],
        (
            ~df_application_summary["has_primary_group"]
            & df_application_summary["has_fallback_division"]
        ),
        df_application_summary["is_unresolved"],
    ],
    [
        "PRIMARY_RESOLVED",
        "FALLBACK_RESOLVED",
        "UNRESOLVED",
    ],
    default="UNKNOWN",
)

print("Application summary rows:", len(df_application_summary))
display(df_application_summary.head())

# Analysis 1 — Classification funnel and overall coverage

## Why analyse this

This is the most important high-level operational measure.

It tells stakeholders:

- how many unique applications entered the pipeline
- how many had usable text
- how many were successfully classified at the preferred 4-digit Group level
- how many required fallback to the less-granular 2-digit Division level
- how many remained unclassified

A high fallback or unresolved rate can indicate:

- domain shift
- insufficient text
- model threshold issues
- weak training coverage for parts of the taxonomy

In [ ]:
total_apps = len(df_application_summary)
usable_text_apps = int(
    df_application_summary["has_any_text"].sum()
)

status_counts = (
    df_application_summary[
        "classification_status"
    ]
    .value_counts()
)

primary_apps = int(
    status_counts.get("PRIMARY_RESOLVED", 0)
)
fallback_apps = int(
    status_counts.get("FALLBACK_RESOLVED", 0)
)
unresolved_apps = int(
    status_counts.get("UNRESOLVED", 0)
)

coverage_summary = pd.DataFrame({
    "metric": [
        "Total unique applications",
        "Applications with usable text",
        "Primary Group resolved",
        "Fallback Division resolved",
        "Still unresolved",
    ],
    "count": [
        total_apps,
        usable_text_apps,
        primary_apps,
        fallback_apps,
        unresolved_apps,
    ],
})

coverage_summary["percentage_of_total"] = (
    coverage_summary["count"]
    / total_apps
    * 100
).round(2)

display(coverage_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

plot_data = (
    df_application_summary[
        "classification_status"
    ]
    .value_counts()
    .reindex(
        [
            "PRIMARY_RESOLVED",
            "FALLBACK_RESOLVED",
            "UNRESOLVED",
        ],
        fill_value=0,
    )
)

plot_data.plot(
    kind="bar",
    ax=ax,
)

ax.set_title("Applications by classification outcome")
ax.set_xlabel("Classification outcome")
ax.set_ylabel("Unique applications")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Analysis 2 — Missing-field impact

## Why analyse this

The model uses only `ApplicationTitle` and `ApplicationSummary`.

We therefore want to know whether classification success changes materially when:

- both fields are available
- only title is available
- only summary is available
- neither is available

This distinguishes model limitations from simple input-data limitations.

If unresolved rates are much higher when one text field is missing, improving data completeness may deliver value without retraining the model.

In [ ]:
df_application_summary["text_availability"] = np.select(
    [
        (
            df_application_summary["has_title"]
            & df_application_summary["has_summary"]
        ),
        (
            df_application_summary["has_title"]
            & ~df_application_summary["has_summary"]
        ),
        (
            ~df_application_summary["has_title"]
            & df_application_summary["has_summary"]
        ),
    ],
    [
        "TITLE_AND_SUMMARY",
        "TITLE_ONLY",
        "SUMMARY_ONLY",
    ],
    default="NO_TEXT",
)

missing_field_analysis = (
    pd.crosstab(
        df_application_summary["text_availability"],
        df_application_summary["classification_status"],
    )
)

missing_field_rates = (
    missing_field_analysis
    .div(
        missing_field_analysis.sum(axis=1),
        axis=0,
    )
    .mul(100)
    .round(2)
)

print("Counts")
display(missing_field_analysis)

print("Row percentages")
display(missing_field_rates)

# Analysis 3 — Text length versus classification outcome

## Why analyse this

Applications with extremely short or generic text may provide too little semantic information for the classifier.

We compare title, summary and total word counts across:

- `PRIMARY_RESOLVED`
- `FALLBACK_RESOLVED`
- `UNRESOLVED`

### What this can tell us

If fallback/unresolved applications have substantially shorter text, the issue may partly be input information quality rather than purely model performance.

If unresolved applications have long, detailed text, that points more strongly toward domain/taxonomy/model coverage issues.

In [ ]:
text_length_summary = (
    df_application_summary
    .groupby("classification_status")
    [
        [
            "title_word_count",
            "summary_word_count",
            "total_text_word_count",
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
        ]
    )
    .round(2)
)

display(text_length_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

box_data = [
    df_application_summary.loc[
        df_application_summary[
            "classification_status"
        ] == status,
        "total_text_word_count",
    ].values
    for status in [
        "PRIMARY_RESOLVED",
        "FALLBACK_RESOLVED",
        "UNRESOLVED",
    ]
]

ax.boxplot(
    box_data,
    labels=[
        "Primary",
        "Fallback",
        "Unresolved",
    ],
    showfliers=False,
)

ax.set_title("Total text length by classification outcome")
ax.set_xlabel("Classification outcome")
ax.set_ylabel("Title + summary word count")
plt.tight_layout()
plt.show()

# Analysis 4 — Multi-label prediction count

## Why analyse this

The primary model is multi-label, so assigning multiple Groups is expected.

However, unusually large numbers of categories may indicate:

- thresholds that are too permissive
- highly interdisciplinary applications
- ambiguous text
- overprediction

We analyse both the normal prediction-count distribution and extreme cases.

In [ ]:
prediction_count_distribution = (
    df_application_summary[
        "prediction_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("prediction_count")
    .reset_index(name="applications")
)

prediction_count_distribution["percentage"] = (
    prediction_count_distribution["applications"]
    / total_apps
    * 100
).round(2)

display(prediction_count_distribution)

print(
    "Mean predictions per application:",
    round(
        df_application_summary[
            "prediction_count"
        ].mean(),
        2,
    ),
)

print(
    "Median predictions per application:",
    df_application_summary[
        "prediction_count"
    ].median(),
)

print(
    f"Applications with {HIGH_PREDICTION_COUNT}+ predictions:",
    int(
        (
            df_application_summary[
                "prediction_count"
            ]
            >= HIGH_PREDICTION_COUNT
        ).sum()
    ),
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

prediction_count_distribution.plot(
    x="prediction_count",
    y="applications",
    kind="bar",
    legend=False,
    ax=ax,
)

ax.set_title("Number of predicted categories per application")
ax.set_xlabel("Predicted categories")
ax.set_ylabel("Applications")
plt.tight_layout()
plt.show()

# Analysis 5 — Score distribution

## Why analyse this

The `score` is marked as `uncalibrated`, so it should **not** be interpreted as a literal probability such as “80% chance of being correct”.

It is still useful as a relative model-confidence signal.

We examine:

- overall score distribution
- primary versus fallback score distributions
- category-level score distributions

### What this can tell us

A large concentration of low selected scores may indicate borderline classifications.

Categories whose selected scores are consistently much lower than others may deserve threshold or training-data review.

In [ ]:
predicted_rows = df_analysis.loc[
    df_analysis["category_id"].notna()
].copy()

score_summary = (
    predicted_rows
    .groupby("prediction_level")["score"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
        ]
    )
    .round(3)
)

display(score_summary)

score_quantiles = (
    predicted_rows["score"]
    .quantile(
        [
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .rename("score")
)

display(score_quantiles)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

predicted_rows["score"].dropna().plot(
    kind="hist",
    bins=30,
    ax=ax,
)

ax.set_title("Distribution of selected model scores")
ax.set_xlabel("Uncalibrated score")
ax.set_ylabel("Prediction rows")
plt.tight_layout()
plt.show()

# Analysis 6 — Primary versus fallback dependency

## Why analyse this

The fallback model deliberately trades granularity for coverage:

- primary success gives a 4-digit Group
- fallback success gives only a 2-digit Division

We want to understand how much the overall system depends on fallback.

### What this tells us

A low fallback rate indicates the Group model covers most applications well.

A high fallback rate indicates that a significant part of production output is only available at Division level, which may motivate:

- improved Group-level training data
- better threshold tuning
- taxonomy-specific model work

In [ ]:
fallback_dependency = pd.DataFrame({
    "classification_status": [
        "PRIMARY_RESOLVED",
        "FALLBACK_RESOLVED",
        "UNRESOLVED",
    ],
    "applications": [
        primary_apps,
        fallback_apps,
        unresolved_apps,
    ],
})

fallback_dependency["percentage"] = (
    fallback_dependency["applications"]
    / total_apps
    * 100
).round(2)

display(fallback_dependency)

if primary_apps + fallback_apps > 0:
    print(
        "Share of classified applications requiring fallback:",
        round(
            fallback_apps
            / (primary_apps + fallback_apps)
            * 100,
            2,
        ),
        "%",
    )

# Analysis 7 — ApplicationOriginSource behaviour

## Why analyse this

The same model may behave differently across source systems if:

- different systems have different text quality
- fields are populated differently
- application domains differ by source
- historical data standards differ

We compare primary, fallback and unresolved rates by `ApplicationOriginSource`.

### What this can tell us

A source with a materially higher fallback/null rate may warrant targeted data-quality investigation before assuming the model itself is the only problem.

In [ ]:
source_analysis = (
    pd.crosstab(
        df_application_summary[
            "ApplicationOriginSource"
        ],
        df_application_summary[
            "classification_status"
        ],
    )
)

source_analysis["TOTAL"] = (
    source_analysis.sum(axis=1)
)

for status in [
    "PRIMARY_RESOLVED",
    "FALLBACK_RESOLVED",
    "UNRESOLVED",
]:
    if status not in source_analysis.columns:
        source_analysis[status] = 0

source_analysis["PRIMARY_%"] = (
    source_analysis["PRIMARY_RESOLVED"]
    / source_analysis["TOTAL"]
    * 100
).round(2)

source_analysis["FALLBACK_%"] = (
    source_analysis["FALLBACK_RESOLVED"]
    / source_analysis["TOTAL"]
    * 100
).round(2)

source_analysis["UNRESOLVED_%"] = (
    source_analysis["UNRESOLVED"]
    / source_analysis["TOTAL"]
    * 100
).round(2)

display(
    source_analysis.sort_values(
        "TOTAL",
        ascending=False,
    )
)

# Analysis 8 — Category concentration and taxonomy coverage

## Why analyse this

A healthy classifier should not necessarily predict every category equally, because the true application population may itself be imbalanced.

However, extreme concentration can reveal:

- dominant classes
- taxonomy areas with little/no coverage
- training imbalance
- classes that are effectively never selected

We analyse Group and Division outputs **separately** because they represent different taxonomy levels.

In [ ]:
primary_category_counts = (
    predicted_rows.loc[
        predicted_rows[
            "prediction_level"
        ] == "PRIMARY_GROUP",
        "category_id",
    ]
    .value_counts()
    .rename_axis("category_id")
    .reset_index(name="prediction_rows")
)

fallback_category_counts = (
    predicted_rows.loc[
        predicted_rows[
            "prediction_level"
        ] == "FALLBACK_DIVISION",
        "category_id",
    ]
    .value_counts()
    .rename_axis("category_id")
    .reset_index(name="prediction_rows")
)

print("Unique primary Group codes predicted:", len(primary_category_counts))
print("Unique fallback Division codes predicted:", len(fallback_category_counts))

print("\nTop primary Groups")
display(primary_category_counts.head(20))

print("\nTop fallback Divisions")
display(fallback_category_counts.head(20))

## Category concentration indicators

The top-category share helps identify whether a small number of classes dominate the model output.

For example, if the top 10 Groups account for a very large percentage of all Group prediction rows, it may be worth comparing that against the expected real-world distribution of FoR categories.

In [ ]:
def top_share(count_df, top_n):
    if len(count_df) == 0:
        return np.nan

    total = count_df["prediction_rows"].sum()

    return (
        count_df.head(top_n)["prediction_rows"].sum()
        / total
        * 100
    )

for n in [5, 10, 20]:
    print(
        f"Top {n} primary Groups share:",
        round(top_share(primary_category_counts, n), 2),
        "%",
    )

for n in [3, 5, 10]:
    print(
        f"Top {n} fallback Divisions share:",
        round(top_share(fallback_category_counts, n), 2),
        "%",
    )

# Analysis 9 — Rare category predictions

## Why analyse this

Categories predicted only a few times can be important for two reasons:

1. They may genuinely be rare research areas.
2. They may represent unstable or unusual classifications that deserve manual review.

This analysis does **not** automatically treat rare predictions as errors. It identifies them as candidates for inspection.

In [ ]:
rare_primary_categories = (
    primary_category_counts.loc[
        primary_category_counts[
            "prediction_rows"
        ] <= RARE_CATEGORY_MAX_APPLICATIONS
    ]
    .copy()
)

print(
    "Primary categories with",
    RARE_CATEGORY_MAX_APPLICATIONS,
    "or fewer prediction rows:",
    len(rare_primary_categories),
)

display(rare_primary_categories.head(50))

# Analysis 10 — Common multi-label Group combinations

## Why analyse this

The Group model is multi-label, so related FoR Groups should frequently appear together.

Looking at common combinations can reveal:

- sensible interdisciplinary patterns
- repeated taxonomy relationships
- suspicious or unexpected category combinations

Only primary 4-digit Group predictions are used here. Fallback Divisions are excluded because they are a different taxonomy level.

In [ ]:
primary_only = df_analysis.loc[
    df_analysis["prediction_level"] == "PRIMARY_GROUP",
    ID_FIELDS + ["category_id"],
].copy()

group_combinations = (
    primary_only
    .groupby(ID_FIELDS)["category_id"]
    .apply(
        lambda x: tuple(
            sorted(
                set(
                    x.dropna().astype(str)
                )
            )
        )
    )
)

multi_group_combinations = (
    group_combinations[
        group_combinations.apply(len) > 1
    ]
    .value_counts()
    .rename_axis("group_combination")
    .reset_index(name="applications")
)

display(multi_group_combinations.head(30))

# Analysis 11 — Score by category

## Why analyse this

Two categories can have similar prediction volumes but very different selected-score distributions.

A category whose predictions consistently have low scores may indicate:

- a difficult decision boundary
- overlapping classes
- insufficient or heterogeneous training examples
- a threshold that deserves review

The results below should be interpreted as **relative score behaviour**, not calibrated probabilities.

In [ ]:
category_score_analysis = (
    predicted_rows
    .groupby(
        [
            "prediction_level",
            "category_id",
        ]
    )["score"]
    .agg(
        prediction_rows="count",
        mean_score="mean",
        median_score="median",
        min_score="min",
        max_score="max",
    )
    .reset_index()
)

category_score_analysis[
    [
        "mean_score",
        "median_score",
        "min_score",
        "max_score",
    ]
] = (
    category_score_analysis[
        [
            "mean_score",
            "median_score",
            "min_score",
            "max_score",
        ]
    ]
    .round(3)
)

print("Lowest mean-score categories with at least 5 predictions")
display(
    category_score_analysis.loc[
        category_score_analysis[
            "prediction_rows"
        ] >= 5
    ]
    .sort_values(
        "mean_score",
        ascending=True,
    )
    .head(30)
)

# Analysis 12 — Unresolved application profiling

## Why analyse this

These are the applications where neither model was able to provide a category.

This is the most useful population for diagnosing future model improvements.

We separate:

- **no-text unresolved** — there was no usable title or summary
- **text-present unresolved** — the model had text but neither primary nor fallback selected a category

### What this tells us

No-text unresolved cases are mainly a data availability problem.

Text-present unresolved cases are stronger candidates for:

- manual semantic review
- new training examples
- threshold review
- taxonomy coverage investigation

In [ ]:
df_unclassified = (
    df_application_summary.loc[
        df_application_summary[
            "classification_status"
        ] == "UNRESOLVED"
    ]
    .copy()
)

df_unclassified["unresolved_type"] = np.where(
    df_unclassified["has_any_text"],
    "TEXT_PRESENT_BUT_UNCLASSIFIED",
    "NO_MODEL_TEXT",
)

unresolved_type_summary = (
    df_unclassified[
        "unresolved_type"
    ]
    .value_counts()
    .rename_axis("unresolved_type")
    .reset_index(name="applications")
)

display(unresolved_type_summary)

print("\nText-present unresolved examples")
display(
    df_unclassified.loc[
        df_unclassified[
            "unresolved_type"
        ] == "TEXT_PRESENT_BUT_UNCLASSIFIED",
        ID_FIELDS
        + [
            "ApplicationTitle",
            "ApplicationSummary",
            "total_text_word_count",
        ]
    ]
    .head(20)
)

# Analysis 13 — Potential anomaly flags

## Why analyse this

This section checks for conditions that should either never occur or deserve manual review.

We flag:

- duplicate application/source/category predictions
- category codes that are not 2 or 4 digits
- null category with non-null score or score type
- predicted category with missing score/score type
- scores outside 0–1
- applications with unusually many categories
- low-score selected predictions

These checks help distinguish genuine modelling behaviour from output-pipeline defects.

In [ ]:
prediction_duplicates = (
    df_predictions.loc[
        df_predictions["category_id"].notna()
    ]
    .duplicated(
        subset=ID_FIELDS + ["category_id"],
        keep=False,
    )
)

invalid_category_format = (
    df_predictions["category_id"]
    .astype("string")
    .notna()
    & ~df_predictions["category_id"]
    .astype("string")
    .str.fullmatch(r"\d{2}|\d{4}", na=False)
)

null_category_inconsistent = (
    df_predictions["category_id"].isna()
    & (
        df_predictions["score_type"].notna()
        | df_predictions["score"].notna()
    )
)

prediction_missing_result = (
    df_predictions["category_id"].notna()
    & (
        df_predictions["score_type"].isna()
        | df_predictions["score"].isna()
    )
)

score_out_of_range = (
    df_predictions["score"].notna()
    & (
        (df_predictions["score"] < 0)
        | (df_predictions["score"] > 1)
    )
)

high_prediction_applications = (
    df_application_summary.loc[
        df_application_summary[
            "prediction_count"
        ] >= HIGH_PREDICTION_COUNT
    ]
    .copy()
)

low_score_predictions = (
    predicted_rows.loc[
        predicted_rows["score"]
        < LOW_SCORE_REVIEW_CUTOFF
    ]
    .copy()
)

anomaly_summary = pd.DataFrame({
    "check": [
        "Duplicate application/source/category prediction rows",
        "Invalid category code format",
        "Null category but non-null score/score_type",
        "Predicted category missing score/score_type",
        "Score outside 0-1",
        f"Applications with {HIGH_PREDICTION_COUNT}+ categories",
        f"Selected predictions below score {LOW_SCORE_REVIEW_CUTOFF}",
    ],
    "count": [
        int(prediction_duplicates.sum()),
        int(invalid_category_format.sum()),
        int(null_category_inconsistent.sum()),
        int(prediction_missing_result.sum()),
        int(score_out_of_range.sum()),
        len(high_prediction_applications),
        len(low_score_predictions),
    ],
})

display(anomaly_summary)

# Analysis 14 — Input/output completeness

## Why analyse this

The stakeholder requirement is that the final output represents the **complete unique input population**, regardless of whether the models could classify an application.

The only intended population reduction is deduplication by:

`ApplicationID + ApplicationOriginSource`

Therefore the expected result here is:

**Applications missing from output = 0**

If it is not zero, there is a pipeline/output completeness issue rather than a model-performance issue.

In [ ]:
input_keys = (
    df_input[ID_FIELDS]
    .drop_duplicates()
)

output_keys = (
    df_predictions[ID_FIELDS]
    .drop_duplicates()
)

completeness_check = input_keys.merge(
    output_keys,
    on=ID_FIELDS,
    how="left",
    indicator=True,
)

missing_from_output = (
    completeness_check.loc[
        completeness_check["_merge"]
        == "left_only"
    ]
)

extra_in_output = (
    output_keys.merge(
        input_keys,
        on=ID_FIELDS,
        how="left",
        indicator=True,
    )
    .loc[
        lambda x:
        x["_merge"] == "left_only"
    ]
)

print("Unique input applications:", len(input_keys))
print("Unique output applications:", len(output_keys))
print("Applications missing from output:", len(missing_from_output))
print("Applications present in output but not input:", len(extra_in_output))

if len(missing_from_output):
    display(missing_from_output.head(20))

if len(extra_in_output):
    display(extra_in_output.head(20))

# Analysis 15 — Compact stakeholder summary

## Why this section exists

The earlier sections are diagnostic and aimed at data scientists.

This section produces a compact set of metrics that can be copied into a stakeholder update without exposing implementation detail.

In [ ]:
stakeholder_summary = pd.DataFrame({
    "metric": [
        "Total unique applications",
        "Applications classified at Group level",
        "Applications classified only at Division fallback level",
        "Applications unresolved",
        "Applications with no usable title/summary",
        "Overall classification coverage",
        "Share of classified applications requiring fallback",
        "Average categories per classified application",
        "Applications with multiple predicted categories",
    ],
    "value": [
        total_apps,
        primary_apps,
        fallback_apps,
        unresolved_apps,
        int(
            (
                df_application_summary[
                    "has_any_text"
                ] == False
            ).sum()
        ),
        (
            round(
                (
                    primary_apps
                    + fallback_apps
                )
                / total_apps
                * 100,
                2,
            )
            if total_apps
            else np.nan
        ),
        (
            round(
                fallback_apps
                / (
                    primary_apps
                    + fallback_apps
                )
                * 100,
                2,
            )
            if (
                primary_apps
                + fallback_apps
            )
            else np.nan
        ),
        round(
            df_application_summary.loc[
                df_application_summary[
                    "prediction_count"
                ] > 0,
                "prediction_count",
            ].mean(),
            2,
        ),
        int(
            (
                df_application_summary[
                    "prediction_count"
                ] > 1
            ).sum()
        ),
    ],
})

display(stakeholder_summary)

## 16. Save useful analysis datasets locally

Three reusable Parquet outputs are created:

### `for_prediction_analysis.parquet`
Prediction-level input + prediction data.

Useful for:
- category-level analysis
- score analysis
- detailed manual inspection

### `for_application_summary.parquet`
One row per unique application.

Useful for:
- classification coverage
- fallback/null analysis
- source and text-quality analysis

### `for_unclassified.parquet`
Only applications unresolved after both models.

Useful for:
- manual review
- identifying potential future training examples
- understanding whether unresolved cases are caused by missing/short text or genuinely difficult applications

No S3 write is performed by this notebook.

In [ ]:
analysis_path = (
    ANALYSIS_OUTPUT_DIR
    / "for_prediction_analysis.parquet"
)

application_summary_path = (
    ANALYSIS_OUTPUT_DIR
    / "for_application_summary.parquet"
)

unclassified_path = (
    ANALYSIS_OUTPUT_DIR
    / "for_unclassified.parquet"
)

df_analysis.to_parquet(
    analysis_path,
    index=False,
    engine="pyarrow",
)

df_application_summary.to_parquet(
    application_summary_path,
    index=False,
    engine="pyarrow",
)

df_unclassified.to_parquet(
    unclassified_path,
    index=False,
    engine="pyarrow",
)

print("Saved:")
print(" -", analysis_path)
print(" -", application_summary_path)
print(" -", unclassified_path)

# Suggested interpretation workflow

When reviewing the results, use this order:

1. **Start with the funnel**  
   Establish the overall Group / fallback / unresolved proportions.

2. **Check missing-field and text-length analysis**  
   Determine how much of the unresolved/fallback problem can be explained by weak input text.

3. **Review source-system differences**  
   Identify whether particular sources have materially different outcomes.

4. **Review multi-label behaviour**  
   Make sure the model is not assigning excessive numbers of categories.

5. **Review category concentration and rare categories**  
   Look for dominant classes, unused areas of the taxonomy and unstable long-tail predictions.

6. **Review score distributions**  
   Identify categories and cases with relatively weak selected scores.

7. **Inspect text-present unresolved applications**  
   These are particularly valuable candidates for model improvement and manual taxonomy review.

8. **Finish with anomaly and completeness checks**  
   Confirm that apparent model issues are not actually output-pipeline defects.

## Important caveat

The current `score` is explicitly labelled `uncalibrated`.  
Use it for relative comparison and diagnostics, but do not describe a score such as `0.80` as an “80% probability of correctness”.